# Laboratorio L16 - Norme in pratica: AI Act, violazioni di dati, minori

B.4 - Cybersicurezza e IA

Tre esercizi in cui una regola giuridica viene tradotta in codice. Il codice è una **semplificazione didattica**: le norme contengono eccezioni, definizioni e condizioni che qui non sono riportate. Non sostituisce la lettura dei testi né un parere legale.

## 1. Classificare sistemi di IA secondo l'AI Act (esercizi 1 e 2)

Prima di eseguire le celle, compilare a mano la scheda: per ciascuno dei dieci sistemi, la categoria (vietato, alto rischio, obblighi di trasparenza, rischio minimo) e il motivo.

Ogni sistema è descritto da un dizionario di caratteristiche vere o false. La funzione `classifica` applica, in ordine, le regole essenziali:

- pratiche vietate (art. 5): riconoscimento delle emozioni a scuola o sul lavoro, punteggio sociale, sfruttamento delle vulnerabilità legate all'età, manipolazione
- alto rischio nell'istruzione (art. 6 e Allegato III, punto 3): ammissione, valutazione dell'apprendimento, orientamento al livello di istruzione, sorveglianza durante le prove
- obblighi di trasparenza (art. 50): sistemi che interagiscono con persone, contenuti sintetici
- altrimenti: rischio minimo

Un sistema ad alto rischio può avere anche obblighi di trasparenza: la funzione li restituisce entrambi.

In [ ]:
def classifica(s):
    """Classificazione semplificata secondo l'AI Act. s: dizionario di caratteristiche."""
    if s.get("emozioni") and s.get("contesto") in ("scuola", "lavoro"):
        return "VIETATO", "art. 5: riconoscimento delle emozioni a scuola o sul lavoro"
    if s.get("punteggio_sociale"):
        return "VIETATO", "art. 5: punteggio sociale"
    if s.get("sfrutta_eta") or s.get("manipolazione"):
        return "VIETATO", "art. 5: manipolazione o sfruttamento di vulnerabilità"
    obblighi = []
    if s.get("interagisce"):
        obblighi.append("dichiarare che si interagisce con un sistema di IA")
    if s.get("contenuti_sintetici"):
        obblighi.append("marcare i contenuti come generati artificialmente")
    usi_istruzione = {"ammissione", "valutazione", "orientamento", "sorveglianza_prove"}
    if s.get("uso") in usi_istruzione:
        return "ALTO RISCHIO", "Allegato III, punto 3 (istruzione)" + ("; trasparenza: " + ", ".join(obblighi) if obblighi else "")
    if obblighi:
        return "TRASPARENZA", "art. 50: " + ", ".join(obblighi)
    return "RISCHIO MINIMO", "nessun obbligo specifico; codici di condotta volontari"

SISTEMI = {
    "correttore automatico dei compiti con voto":     {"uso": "valutazione"},
    "sorveglianza delle prove online con webcam":     {"uso": "sorveglianza_prove"},
    "chatbot di segreteria":                          {"interagisce": True},
    "rilevatore dell'attenzione tramite webcam":      {"emozioni": True, "contesto": "scuola"},
    "generatore di immagini per il giornalino":       {"contenuti_sintetici": True},
    "filtro antispam della posta della scuola":       {},
    "sistema che ordina le domande di iscrizione":    {"uso": "ammissione"},
    "app di orientamento che consiglia l'indirizzo":  {"uso": "orientamento", "interagisce": True},
    "punteggio di comportamento con premi e divieti": {"punteggio_sociale": True},
    "correttore ortografico nel programma di scrittura": {},
}

for nome, caratteristiche in SISTEMI.items():
    categoria, motivo = classifica(caratteristiche)
    print(f"{nome:50} {categoria:15} {motivo}")

**Domande**

- In quali casi la vostra classificazione a mano era diversa? Chi ha ragione, e perché?
- Il "punteggio di comportamento con premi e divieti" è vietato in ogni caso? Che cosa dice l'art. 5 sulle conseguenze del punteggio? (Leggere il testo, link nella lezione.)
- Aggiungere un sistema descritto da voi e prevederne la classificazione prima di eseguire la cella.

Risposte:

In [ ]:
# esercizio 2: aggiungere un sistema
SISTEMI_NUOVI = {
    "assistente vocale che legge le circolari con voce sintetica": {"contenuti_sintetici": True, "interagisce": True},
}
for nome, caratteristiche in SISTEMI_NUOVI.items():
    print(nome, "->", classifica(caratteristiche))

## 2. Violazione di dati personali: i tempi di notifica (esercizio 3)

Il GDPR (art. 33) prevede che il titolare notifichi una violazione di dati personali al Garante entro 72 ore da quando ne è venuto a conoscenza, salvo che sia improbabile un rischio per le persone. Se il rischio è elevato, deve anche comunicarla agli interessati senza ingiustificato ritardo (art. 34).

- `datetime(anno, mese, giorno, ora, minuti)`: un istante
- `timedelta(hours=72)`: un intervallo di 72 ore; sommato a un istante dà la scadenza

In [ ]:
from datetime import datetime, timedelta

def valuta_violazione(scoperta, rischio):
    """rischio: 'improbabile', 'presente', 'elevato'"""
    print("violazione scoperta il", scoperta.strftime("%d/%m/%Y alle %H:%M"))
    if rischio == "improbabile":
        print("  notifica al Garante non obbligatoria; documentare comunque la violazione nel registro interno")
        return
    print("  notificare al Garante entro:", (scoperta + timedelta(hours=72)).strftime("%d/%m/%Y alle %H:%M"))
    if rischio == "elevato":
        print("  comunicare la violazione anche agli interessati, senza ingiustificato ritardo")

# caso: un file con i voti e i recapiti di una classe è stato inviato per errore a un indirizzo esterno
valuta_violazione(datetime(2026, 11, 13, 17, 30), "presente")

**Domande**

- Se la violazione viene scoperta un venerdì pomeriggio, la scadenza cade nel fine settimana? Il GDPR prevede eccezioni per i giorni festivi?
- Il file inviato per errore conteneva anche certificati medici. Il rischio resta "presente"? Rieseguire con la valutazione che ritenete corretta.

Risposte:

## 3. Minori e sistemi di IA (esercizio 4)

In Italia:

- il consenso ai servizi della società dell'informazione può essere dato autonomamente dai 14 anni (d.lgs. 196/2003, art. 2-quinquies)
- la legge 132/2025 (art. 4) prevede che i minori di 14 anni possano accedere ai sistemi di IA solo con il consenso di chi esercita la responsabilità genitoriale

Le condizioni d'uso dei singoli servizi possono fissare età minime più alte. La funzione calcola l'età a una data e indica che cosa serve. Le date di nascita sono inventate.

In [ ]:
from datetime import date

def eta(nascita, oggi):
    return oggi.year - nascita.year - ((oggi.month, oggi.day) < (nascita.month, nascita.day))

def requisiti_ia(nascita, oggi, eta_minima_servizio=None):
    anni = eta(nascita, oggi)
    esito = "consenso dei genitori necessario (legge 132/2025)" if anni < 14 else "può acconsentire autonomamente"
    if eta_minima_servizio and anni < eta_minima_servizio:
        esito += f"; ma le condizioni del servizio richiedono almeno {eta_minima_servizio} anni"
    return anni, esito

oggi = date(2026, 10, 1)
for nascita in [date(2013, 3, 15), date(2012, 9, 30), date(2012, 10, 2), date(2010, 1, 1)]:
    print(nascita, requisiti_ia(nascita, oggi, eta_minima_servizio=13))

**Domande**

- Perché la funzione `eta` non calcola semplicemente `oggi.year - nascita.year`? Verificare con il terzo caso.
- Un servizio di IA usato dalla scuola con account istituzionali: chi è il titolare del trattamento dei dati degli studenti?

Risposte: